## Part 1

### Task 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from numpy.linalg import inv, matrix_power

# --- Task 1: Setup and Basic Simulation ---
# Transition matrix P (1 time step = 1 month)
P = np.array([
    [0.9915, 0.005,  0.0025, 0,      0.001 ],
    [0,      0.986,  0.005,  0.004,  0.005 ],
    [0,      0,      0.992,  0.003,  0.005 ],
    [0,      0,      0,      0.991,  0.009 ],
    [0,      0,      0,      0,      1.0   ]
])

def simulate_woman(P):
    """Simulates a single woman's path until death (state 5, index 4)."""
    state = 0
    path = [state]
    while state != 4:
        state = np.random.choice(5, p=P[state])
        path.append(state)
    return path

np.random.seed(42)
n_sims = 1000
lifetimes = []
local_recurrence_count = 0
states_at_120 = []

for _ in range(n_sims):
    path = simulate_woman(P)
    lifetimes.append(len(path) - 1)
    
    if 1 in path or 3 in path:
        local_recurrence_count += 1
        
    # State at t = 120
    if len(path) - 1 >= 120:
        states_at_120.append(path[120])
    else:
        states_at_120.append(4) # Already absorbed in death state

# Plotting the lifetime distribution (Task 1)
plt.figure()
plt.hist(lifetimes, bins=30, density=False, alpha=0.6, color='skyblue', edgecolor='black', label='Simulated')
plt.title('Lifetime Distribution after Surgery (Task 1)')
plt.xlabel('Months')
plt.ylabel('Density')

# Proportion of local recurrence
prop_local = local_recurrence_count / n_sims
print(f"Task 1: Proportion of women with eventual local recurrence: {prop_local:.4f}")

### Task 2

In [ ]:
# --- Task 2: Distribution at t = 120 ---
# Vector representing 100% probability of starting in State 1 (index 0)
pi_0 = np.array([1.0, 0.0, 0.0, 0.0, 0.0])

p_120_theory = pi_0 @ matrix_power(P, 120)
expected_counts = p_120_theory * n_sims

observed_counts = [states_at_120.count(i) for i in range(5)]
chi2_stat, p_val = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print(f"\nTask 2: Observed counts at t=120: {observed_counts}")
print(f"Task 2: Expected counts at t=120: {np.round(expected_counts, 1)}")
print(f"Task 2: Chi-square statistic: {chi2_stat:.4f}, p-value: {p_val:.3f}")

### Task 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from numpy.linalg import inv, matrix_power

# 1. Setup Phase-Type Matrices
Ps = P[:4, :4]
ps = P[:4, 4]
pi_s = np.array([1.0, 0.0, 0.0, 0.0])
I = np.eye(4)
N = inv(I - Ps)
e = np.ones(4)


mean_theory = pi_s @ N @ e

# Theoretical PMF and CDF
t_max = max(lifetimes)
t_vals = np.arange(1, t_max + 1)
pmf_theory = np.array([pi_s @ matrix_power(Ps, t-1) @ ps for t in t_vals])




mean_sim = np.mean(lifetimes)
std_sim = np.std(lifetimes)


bin_edges = np.linspace(1, t_max, 16)
obs_counts, _ = np.histogram(lifetimes, bins=bin_edges)

# Compute expected probabilities for each bin from theoretical PMF
exp_counts = []
for i in range(len(bin_edges) - 1):
    low, high = int(np.floor(bin_edges[i])), int(np.floor(bin_edges[i+1]))

    prob = np.sum(pmf_theory[(t_vals >= low) & (t_vals < high)])
    exp_counts.append(prob * n_sims)

exp_counts = np.array(exp_counts)

exp_counts = exp_counts * (np.sum(obs_counts) / np.sum(exp_counts))

chi2_stat_fit, p_val_fit = stats.chisquare(f_obs=obs_counts, f_exp=exp_counts)


print("="*60)
print(f"{'STATISTIC':<30} | {'SIMULATED':<12} | {'THEREOTICAL':<12}")
print("="*60)
print(f"{'Mean Lifetime (Months)':<30} | {mean_sim:<12.2f} | {mean_theory:<12.2f}")

print("-"*60)
print(f"Global Chi-Square Goodness-of-Fit Test (15 bins):")
print(f"  - Chi2 Statistic: {chi2_stat_fit:.4f}")
print(f"  - p-value: {p_val_fit:.4f} (A high p-value means the distributions match perfectly!)")
print("="*60)

# Prepare bin labels for the x-axis using the given scheme
bin_labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-2)] + [f"{int(bin_edges[-2])}+"]
x = np.arange(len(bin_labels))
width = 0.42

fig, ax = plt.subplots()

observed = obs_counts
expected = exp_counts

ax.bar(x - width/2, observed, width=width, color="#4C78A8", edgecolor="black", label="Observed")
ax.bar(x + width/2, expected, width=width, color="#F58518", edgecolor="black", alpha=0.85, label="Expected (theory)")

ax.set_title("Lifetimes", pad=10)
ax.set_xlabel("Months")
ax.set_ylabel("Number of women")
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha="center")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

### Task 4

In [ ]:
np.random.seed(42)

accepted_lifetimes = []
while len(accepted_lifetimes) < 1000:
    path = simulate_woman(P)
    
    # 1. Condition: Survived at least 12 months
    survived_12 = (len(path) - 1 >= 12)
    
    # 2. Condition: Cancer reappeared at any point in months 1 to 12 
    early_recurrence = any(state in [1, 2, 3] for state in path[1:13])
    
    if survived_12 and early_recurrence:
        accepted_lifetimes.append(len(path) - 1)

print(f"Task 4: Expected lifetime of a woman with early recurrence: {np.mean(accepted_lifetimes):.2f} months")

### Task 5

In [ ]:
np.random.seed(42)

n_iters = 100
n_women = 200

Y_vals, X_vals = [], []

for _ in range(n_iters):
    batch_lifetimes = [len(simulate_woman(P)) - 1 for _ in range(n_women)]
    # Y = fraction dying within 350 months (our target estimator)
    Y_vals.append(np.mean(np.array(batch_lifetimes) <= 350))
    # X = mean lifetime of the batch (our control variate)
    X_vals.append(np.mean(batch_lifetimes))

Y_vals = np.array(Y_vals)
X_vals = np.array(X_vals)

# Compute optimal control coefficient c*
cov_matrix = np.cov(X_vals, Y_vals)
c_star = -cov_matrix[0, 1] / cov_matrix[0, 0]

# Controlled estimate
# Y_c = Y + c* * (X - E[X])
Y_c = Y_vals + c_star * (X_vals - mean_theory)

var_crude = np.var(Y_vals, ddof=1)
var_cv = np.var(Y_c, ddof=1)
reduction = (1 - (var_cv / var_crude)) * 100

print(f"Task 5: Crude MC estimate P(T <= 350): {np.mean(Y_vals):.4f} (Var: {var_crude:.6f})")
print(f"Task 5: Control Variate estimate:      {np.mean(Y_c):.4f} (Var: {var_cv:.6f})")
print(f"Task 5: Variance reduction achieved:   {reduction:.2f}%")

## Part 2

### Task 7

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.linalg import expm


# Define the transition-rate matrix Q (without the absorbing death state row since we never leave it)
Q = np.array([
    [-0.0085, 0.005,  0.0025, 0,      0.001],
    [0,      -0.014,  0.005,  0.004,  0.005],
    [0,      0,      -0.008,  0.003,  0.005],
    [0,      0,      0,      -0.009,  0.009],
])

def simulate_ctmc(Q_matrix):
    """Simulates a single woman's path in continuous time until death (state 4)."""
    state = 0
    t = 0.0
    path = [(t, state)]
    
    while state != 4:
        # Rate of leaving the current state
        rate = -Q_matrix[state, state]
        
        # Sojourn time is exponentially distributed
        sojourn = np.random.exponential(1.0 / rate)
        t += sojourn
        
        # Calculate transition probabilities to next state
        probs = Q_matrix[state, :].copy()
        probs[state] = 0.0
        probs = probs / rate # Normalize to sum to 1
        
        state = np.random.choice(5, p=probs)
        path.append((t, state))
        
    return path

np.random.seed(42)
n_sims = 1000
lifetimes = []
distant_by_30_5 = 0

for _ in range(n_sims):
    path = simulate_ctmc(Q)
    lifetimes.append(path[-1][0]) 
    
    for time, state in path:
        if time >= 30.5 and state in [2, 3]: 
            distant_by_30_5 += 1
            break

lifetimes = np.array(lifetimes)
mean_lt = np.mean(lifetimes)
std_lt = np.std(lifetimes, ddof=1)

# 95% Confidence Intervals
# For mean: mean +/- 1.96 * (std / sqrt(n))
ci_mean = (mean_lt - 1.96 * std_lt / np.sqrt(n_sims), 
           mean_lt + 1.96 * std_lt / np.sqrt(n_sims))

# For std dev: std +/- 1.96 * (std / sqrt(2n))
se_std = std_lt / np.sqrt(2 * n_sims)
ci_std = (std_lt - 1.96 * se_std, std_lt + 1.96 * se_std)


print(f"Task 7: Mean Lifetime: {mean_lt:.2f} months, 95% CI: [{ci_mean[0]:.2f}, {ci_mean[1]:.2f}]")
print(f"Task 7: Std Dev of Lifetime: {std_lt:.2f} months, 95% CI: [{ci_std[0]:.2f}, {ci_std[1]:.2f}]")
print(f"Task 7: Proportion with distant metastasis by 30.5 months: {distant_by_30_5 / n_sims:.4f}\n")

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(lifetimes, bins=30, density=True, alpha=0.6, color='skyblue', edgecolor='black')
plt.title('Task 7: Lifetime Distribution after Surgery (CTMC)')
plt.xlabel('Months')
plt.ylabel('Density')
plt.show()

### Task 8

In [ ]:
np.random.seed(42)
Qs = Q[:4, :4]

p0 = np.array([1.0, 0.0, 0.0, 0.0])
ones = np.ones(4)

t_vals = np.linspace(0, max(lifetimes), 200)

theoretical_survival = [p0 @ expm(Qs * t) @ ones for t in t_vals]

sorted_lifetimes = np.sort(lifetimes)
empirical_survival = 1.0 - np.arange(1, n_sims + 1) / n_sims

### Task 9

In [ ]:
np.random.seed(42)
Q_treated = np.array([
    [-0.00475, 0.0025, 0.00125, 0,     0.001],
    [0,       -0.006,  0.002,   0.004, 0    ], 
    [0,       0,      -0.008,   0.003, 0.005],
    [0,       0,      0,       -0.009, 0.009]
])

lifetimes_treated = np.array([simulate_ctmc(Q_treated)[-1][0] for _ in range(n_sims)])
sorted_treated = np.sort(lifetimes_treated)
survival_treated = 1.0 - np.arange(1, n_sims + 1) / n_sims

plt.figure(figsize=(10, 6))
plt.step(sorted_lifetimes, empirical_survival, label='Untreated (Simulated)', where='post', color='blue')
plt.step(sorted_treated, survival_treated, label='Treated (Simulated)', where='post', color='red')
plt.title('Kaplan-Meier Survival Curves')
plt.xlabel('Months')
plt.ylabel('Survival Probability S(t)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


print(f"Mean Lifetime (Untreated): {np.mean(lifetimes):.2f} months")
print(f"Mean Lifetime (Treated):   {np.mean(lifetimes_treated):.2f} months")
print(f"Treatment increases mean survival by: {np.mean(lifetimes_treated) - np.mean(lifetimes):.2f} months!")

## Part 3

### Task 12

In [ ]:
import numpy as np


Q_treated = np.array([
    [-0.00475, 0.0025, 0.00125, 0,     0.001],
    [0,       -0.006,  0.002,   0.004, 0    ], 
    [0,       0,      -0.008,   0.003, 0.005],
    [0,       0,      0,       -0.009, 0.009],
    [0,       0,      0,       0,      0]
])

def simulate_ctmc_path(Q_matrix):
    """Simulates a continuous path and records (time, state) tuples."""
    state = 0
    t = 0.0
    path = [(t, state)]
    
    while state != 4:
        rate = -Q_matrix[state, state]
        sojourn = np.random.exponential(1.0 / rate)
        t += sojourn
        
        probs = Q_matrix[state, :].copy()
        probs[state] = 0.0
        probs = probs / rate 
        
        state = np.random.choice(5, p=probs)
        path.append((t, state))
    return path

def observe_screenings(path, interval=48):
    """Takes a snapshot of the patient's state every 48 months."""
    observed_Y = []
    obs_time = 0.0
    current_idx = 0
    
    while True:
        while current_idx < len(path) - 1 and path[current_idx + 1][0] <= obs_time:
            current_idx += 1
            
        state_at_obs = path[current_idx][1]
        observed_Y.append(state_at_obs)
        
        if state_at_obs == 4:
            break
        obs_time += interval
        
    return observed_Y

np.random.seed(42)
n_women = 1000 
time_series_data = []

for _ in range(n_women):
    continuous_path = simulate_ctmc_path(Q_treated)
    time_series_data.append(observe_screenings(continuous_path, interval=48))

print(f"Task 12: Successfully generated screening data for {n_women} women.")


### Task 13

In [ ]:
np.random.seed(42)
def simulate_bridge(start_state, end_state, interval, Q_mat):
    """
    E-Step helper: Rejection sampling to simulate a path that strictly 
    matches the start and end states over the 48-month interval.
    """
    while True:
        t = 0.0
        state = start_state
        path = [(t, state)]
        
        while t < interval and state != 4:
            rate = -Q_mat[state, state]
            if rate == 0:
                break
                
            sojourn = np.random.exponential(1.0 / rate)
            if t + sojourn > interval:
                break 
                
            t += sojourn
            probs = Q_mat[state, :].copy()
            probs[state] = 0.0
            probs = probs / rate
            
            state = np.random.choice(5, p=probs)
            path.append((t, state))
            

        if state == end_state:
            path.append((interval, state))
            return path


Q_k = np.array([
    [-0.03, 0.01, 0.01, 0.00, 0.01],
    [0.00, -0.03, 0.01, 0.01, 0.01],
    [0.00, 0.00, -0.02, 0.01, 0.01],
    [0.00, 0.00, 0.00, -0.01, 0.01],
    [0.00, 0.00, 0.00, 0.00, 0.00]
])

n_iter_max = 20

for k in range(n_iter_max):
    N_counts = np.zeros((5, 5))
    S_times = np.zeros(5)
    
    for Y_i in time_series_data:
        for step in range(len(Y_i) - 1):
            start_s = Y_i[step]
            end_s = Y_i[step + 1]
            
            if start_s == 4:
                continue 
                
            path = simulate_bridge(start_s, end_s, 48, Q_k)
            
            for p_idx in range(len(path) - 1):
                t1, s1 = path[p_idx]
                t2, s2 = path[p_idx + 1]
                
                S_times[s1] += (t2 - t1)
                if s1 != s2: 
                    N_counts[s1, s2] += 1
                    
    Q_next = np.zeros((5, 5))
    for i in range(4):
        for j in range(5):
            if i != j and S_times[i] > 0:
                Q_next[i, j] = N_counts[i, j] / S_times[i]
        
        Q_next[i, i] = -np.sum(Q_next[i, :])
        
    max_diff = np.max(np.abs(Q_next - Q_k))
    print(f"Iteration {k+1}: Max Difference = {max_diff:.6f}")
    
    Q_k = Q_next.copy()
    if max_diff < 1e-3:
        print("Convergence criterion reached!")
        break

print("\nFinal Estimated Q Matrix:")
with np.printoptions(precision=5, suppress=True):
    print(Q_k)
    print()
    print(Q_treated)